# Sandbox tools over MCP

This notebook deploys `pk sandbox mcp`, the prokube SDK's MCP server for sandboxes, in your workspace and gives a [Pydantic AI](https://ai.pydantic.dev/) agent its tools through Agent Gateway. The agent creates a sandbox, writes and runs code in it and deletes it, all over MCP: the notebook contains no sandbox code. See the [README](./README.md) for the same tools in Claude Code and opencode on a laptop.

MCP servers and sandboxes must be enabled on your prokube cluster.

## Setup

In [ ]:
%pip install -q "pydantic-ai-slim[openai,mcp]"

## Deploy the MCP server

[`sandbox-mcp.yaml`](./sandbox-mcp.yaml) runs the server as a ToolHive `MCPServer` with the workspace's identity for sandboxes, so it needs no login:

In [ ]:
!kubectl apply -f sandbox-mcp.yaml
!kubectl wait --for=condition=Ready mcpservers.toolhive.stacklok.dev/sandbox-example --timeout=3m

Labs reach the MCP servers of their workspace through the internal Agent Gateway route without an API key:

In [ ]:
with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()

mcp_url = f"http://agentgateway-proxy.agentgateway-system.svc.cluster.local/_platform/mcp/{namespace}"
print(mcp_url)

## Choose a model

[`mock_model.py`](./mock_model.py) is a stand-in that makes fixed tool calls, so the notebook runs without a model. For a real agent, use any OpenAI-compatible endpoint with tool calling, for example a model from prokube LLM Serving; Pydantic AI then reads `OPENAI_BASE_URL` and `OPENAI_API_KEY`:

In [ ]:
from mock_model import scripted_model

# model = "openai:<model>"
model = scripted_model

## The agent

The agent gets the server's tools as a toolset:

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset

agent = Agent(
    model,
    toolsets=[MCPToolset(mcp_url)],
    system_prompt="You work in prokube sandboxes through your tools. Answer briefly.",
)

In [ ]:
result = await agent.run(
    "Create a sandbox named mcp-demo. In it, write a Python function that returns "
    "the primes below a number, test it, and tell me the primes below 50. "
    "Delete the sandbox when you are done."
)
print(result.output)

The tool calls the agent made:

In [ ]:
for message in result.all_messages():
    for part in message.parts:
        if part.part_kind == "tool-call":
            print(part.tool_name, part.args)

## Clean up

Delete the MCP server:

In [ ]:
!kubectl delete -f sandbox-mcp.yaml